# ECG code-along: signal to features to clusters

Day 2 practice, part A, about 60 minutes.

We go from a raw waveform to per-beat features to an unsupervised split between normal and
abnormal beats, with no labels anywhere. This is the shape of most real biosignal work.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.signal import butter, filtfilt, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans


def data_dir():
    """The data/ folder of this repository, wherever the notebook is opened from."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data").is_dir():
            return p / "data"
    raise FileNotFoundError("Could not find the data folder. Run this notebook from inside the cloned repository.")


DATA = data_dir()
RANDOM_STATE = 42

fs = 360  # samples per second
ecg = np.load(DATA / "ecg_mitbih_208.npy")
print(f"{len(ecg)} samples = {len(ecg)/fs:.0f} seconds of MIT-BIH record 208")

## 1. Look at the raw signal first

In [ ]:
# TODO: plot 10 seconds of raw ECG with a time axis in seconds
# TODO: name the three artefacts you can see (baseline wander, mains hum, sensor noise)

## 2. Filter and detect R peaks

A fixed prominence threshold only works for one recording amplitude. Derive it from the
signal instead, so the same code survives a different lead, a different device, or the
fallback record.

In [ ]:
# TODO: band-pass filter 0.5 to 40 Hz with butter(3, ...) + filtfilt
# TODO: why filtfilt and not lfilter? (hint: phase, and where the R peak lands)

In [ ]:
# TODO: set prominence adaptively, e.g. 0.5 * np.percentile(np.abs(filtered), 99)
# TODO: find_peaks(filtered, distance=int(0.3 * fs), prominence=prominence)
# TODO: overlay detected peaks on 10 seconds of signal and check every beat by eye

## 3. Rhythm features: heart rate and HRV

SDNN and RMSSD are defined on normal-to-normal intervals. Compute them over all beats first,
then over normal beats only once you have the clusters, and compare. The difference is the
point of this section.

In [ ]:
# TODO: RR intervals from the peak positions, in seconds
# TODO: drop physiologically impossible intervals (outside 0.3 to 2.0 s)
# TODO: mean HR, SDNN in ms, RMSSD in ms
# TODO: plot the RR tachogram. Do the outliers come in pairs (short then long)?

## 4. Morphology features: segment every beat

Window: 250 ms before the R peak to 450 ms after, so one window holds P, QRS and T.

In [ ]:
# TODO: build a beats matrix of shape (n_beats, window_length)
# TODO: keep rr_before and rr_after alongside each beat, they carry the prematurity signal
# TODO: plot 20 overlaid beats

## 5. Cluster the beats

In [ ]:
# TODO: standardise, PCA to 5 components, KMeans with 2 clusters
# TODO: plot the mean waveform of each cluster with its size in the legend
# TODO: report the share of beats in the smaller cluster

## 6. Interpretation

1. Which cluster looks abnormal, and which three pieces of evidence support that?
   (QRS width, R amplitude, the RR interval before and after.)
2. Recompute SDNN and RMSSD using normal-to-normal intervals only. How much did they change?
   Which number would you put in a report?
3. k-means found a split. What does it not tell you, and what would you need to call these
   beats ventricular in a clinical report?
4. Your detector missed a beat. Trace what that single miss does to RR, to HRV, and to the
   beats matrix.